# Pipeline Gold : Silver → Delta Lake (Données Agrégées)

## Objectif
Implémenter la troisième étape de l'architecture Médaillon : **Gold** (données agrégées)

1. Lire les données nettoyées depuis Delta Lake Silver
2. Appliquer des **agrégations temporelles** (fenêtres horaires, moyennes)
3. Calculer des **statistiques par dimension** (bâtiment, capteur)
4. Créer des **métriques métier** (KPIs, tendances)
5. Écrire les données agrégées dans Delta Lake (niveau Gold)

## Architecture Médaillon

```
Delta Lake Silver (données nettoyées)
    ↓
GOLD (ce notebook) : Agrégations & Analytics
    ↓
Dashboards & Rapports
```

## Principe Gold

**Gold = Données agrégées** : Optimisées pour l'analyse et la consommation
- ✅ Agrégations temporelles (moyennes horaires, quotidiennes)
- ✅ Agrégations par dimension (par bâtiment, par capteur)
- ✅ Métriques calculées (tendances, anomalies)
- ✅ Schémas dénormalisés pour performance
- ✅ Optimisations pour les requêtes analytiques

## Cas d'usage SmartTech

Les données Gold alimentent :
- Tableaux de bord exécutifs
- Rapports de consommation énergétique
- Statistiques par bâtiment
- Détection d'anomalies agrégées
- Métriques de performance

## 1. Configuration de l'environnement Spark

In [ ]:
# Nettoyage des données Delta Lake Gold et checkpoints (optionnel)
import shutil
import os
from pathlib import Path

# Définir les chemins si pas encore définis
if 'DELTA_SILVER_PATH' not in globals():
    DELTA_SILVER_PATH = os.getenv("DELTA_SILVER_PATH", "/opt/spark/delta/silver")
if 'DELTA_GOLD_PATH' not in globals():
    DELTA_GOLD_PATH = os.getenv("DELTA_GOLD_PATH", "/opt/spark/delta/gold")
if 'CHECKPOINT_GOLD_PATH' not in globals():
    CHECKPOINT_GOLD_PATH = os.getenv("CHECKPOINT_GOLD_PATH", "/opt/spark/checkpoints/gold")

print("🧹 Nettoyage des données Delta Lake Gold et checkpoints...")
print("⚠️  ATTENTION : Cette opération supprime toutes les données existantes !")
print(f"   - Delta Gold : {DELTA_GOLD_PATH}")
print(f"   - Checkpoints : {CHECKPOINT_GOLD_PATH}")

# Nettoyage actif (mettez True pour nettoyer)
CLEAN_GOLD = False  # ⚠️ Changez à True pour nettoyer

if CLEAN_GOLD:
    try:
        # Arrêter toutes les queries en cours si elles existent
        try:
            if 'query_gold' in globals():
                query_gold.stop()
                print("✓ Query Gold arrêtée")
        except:
            pass
        
        # Supprimer la table Delta Gold
        gold_path = Path(DELTA_GOLD_PATH)
        if gold_path.exists():
            shutil.rmtree(str(gold_path))
            print(f"✓ Table Delta Gold supprimée : {DELTA_GOLD_PATH}")
        else:
            print(f"ℹ️  Table Delta Gold n'existe pas encore : {DELTA_GOLD_PATH}")
        
        # Supprimer les checkpoints
        checkpoint_path = Path(CHECKPOINT_GOLD_PATH)
        if checkpoint_path.exists():
            shutil.rmtree(str(checkpoint_path))
            print(f"✓ Checkpoints supprimés : {CHECKPOINT_GOLD_PATH}")
        else:
            print(f"ℹ️  Checkpoints n'existent pas encore : {CHECKPOINT_GOLD_PATH}")
        
        print("\n✅ Nettoyage terminé - Vous pouvez repartir de zéro")
        print("💡 N'oubliez pas de remettre CLEAN_GOLD = False après le nettoyage")
    except Exception as e:
        print(f"❌ Erreur lors du nettoyage : {e}")
else:
    print("\n💡 Pour effectuer le nettoyage, mettez CLEAN_GOLD = True et réexécutez cette cellule")

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *
import os

# Configuration des chemins depuis les variables d'environnement (avec valeurs par défaut)
DELTA_SILVER_PATH = os.getenv("DELTA_SILVER_PATH", "/opt/spark/delta/silver")
DELTA_GOLD_PATH = os.getenv("DELTA_GOLD_PATH", "/opt/spark/delta/gold")
CHECKPOINT_GOLD_PATH = os.getenv("CHECKPOINT_GOLD_PATH", "/opt/spark/checkpoints/gold")

# Configuration Spark depuis les variables d'environnement
SPARK_APP_NAME = os.getenv("SPARK_APP_NAME", "SmartTech-Gold-Pipeline")

# Créer la session Spark avec support Delta Lake
builder = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master(os.getenv("SPARK_MASTER", "local[*]"))

# Utiliser configure_spark_with_delta_pip() qui utilise les JARs installés par pip
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("✓ Spark Session créée avec succès")
print(f"✓ Version Spark : {spark.version}")
print(f"✓ DELTA_SILVER_PATH : {DELTA_SILVER_PATH}")
print(f"✓ DELTA_GOLD_PATH : {DELTA_GOLD_PATH}")
print(f"✓ CHECKPOINT_GOLD_PATH : {CHECKPOINT_GOLD_PATH}")

## 2. Vérification des données Silver

In [ ]:
# Vérifier que les données Silver existent
print("🔍 Vérification des données Silver...")

try:
    silver_df = spark.read.format("delta").load(DELTA_SILVER_PATH)
    silver_count = silver_df.count()
    
    if silver_count > 0:
        print(f"✓ {silver_count} enregistrements trouvés dans Silver")
        print(f"\n📊 Schéma Silver :")
        silver_df.printSchema()
        print(f"\n📋 Aperçu des données Silver :")
        silver_df.show(5, truncate=False)
    else:
        print("⚠️  Aucune donnée trouvée dans Silver")
        print("💡 Exécutez d'abord le notebook 02_pipeline_silver.ipynb")
except Exception as e:
    print(f"❌ Erreur lors de la lecture de Silver : {e}")
    print(f"💡 Vérifiez que Silver existe : {DELTA_SILVER_PATH}")

## 3. Lecture du flux Silver (Streaming)

In [ ]:
# Lire Silver en streaming pour traiter les nouvelles données au fur et à mesure
# Note : On pourrait aussi lire en batch, mais le streaming permet de mettre à jour
# les agrégations en temps réel

silver_stream = spark.readStream \
    .format("delta") \
    .load(DELTA_SILVER_PATH)

print("✓ Source Silver configurée (streaming)")
print(f"\n📊 Schéma du flux Silver :")
silver_stream.printSchema()

## 4. Agrégations temporelles (Fenêtres horaires)

In [ ]:
# Agrégations par fenêtre temporelle (1 heure)
# Utilisation de windowing avec watermark pour gérer les données tardives

# Watermark : données jusqu'à 1 heure de retard acceptées
silver_with_watermark = silver_stream \
    .withWatermark("timestamp", "1 hour")

# Agrégations par fenêtre de 1 heure et par bâtiment
hourly_stats = silver_with_watermark \
    .groupBy(
        window(col("timestamp"), "1 hour"),  # Fenêtre de 1 heure
        col("building_id")
    ) \
    .agg(
        # Température
        avg("temperature").alias("avg_temperature"),
        min("temperature").alias("min_temperature"),
        max("temperature").alias("max_temperature"),
        # Humidité
        avg("humidity").alias("avg_humidity"),
        min("humidity").alias("min_humidity"),
        max("humidity").alias("max_humidity"),
        # Consommation énergétique
        sum("energy_consumption").alias("total_energy_consumption"),
        avg("energy_consumption").alias("avg_energy_consumption"),
        # Compteurs
        count("*").alias("measurement_count"),
        countDistinct("sensor_id").alias("sensor_count"),
        # Anomalies
        sum(when(col("anomaly_detected") == True, 1).otherwise(0)).alias("anomaly_count")
    ) \
    .withColumn("window_start", col("window.start")) \
    .withColumn("window_end", col("window.end")) \
    .drop("window")

print("✓ Agrégations horaires calculées")
print(f"\n📊 Schéma des agrégations horaires :")
hourly_stats.printSchema()

## 5. Statistiques par capteur

In [ ]:
# Agrégations par capteur (dernières valeurs et tendances)
# Utilisation de windowing pour calculer les stats par capteur sur une fenêtre glissante

sensor_stats = silver_with_watermark \
    .groupBy(
        window(col("timestamp"), "1 hour", "30 minutes"),  # Fenêtre glissante de 1h, slide 30min
        col("sensor_id"),
        col("building_id"),
        col("sensor_type")
    ) \
    .agg(
        # Dernières valeurs
        max("timestamp").alias("last_measurement_time"),
        # Température
        avg("temperature").alias("avg_temperature"),
        max("temperature").alias("max_temperature"),
        min("temperature").alias("min_temperature"),
        # Humidité
        avg("humidity").alias("avg_humidity"),
        # Consommation
        sum("energy_consumption").alias("total_energy"),
        avg("energy_consumption").alias("avg_energy"),
        # Compteurs
        count("*").alias("measurement_count"),
        # Anomalies
        sum(when(col("anomaly_detected") == True, 1).otherwise(0)).alias("anomaly_count")
    ) \
    .withColumn("window_start", col("window.start")) \
    .withColumn("window_end", col("window.end")) \
    .drop("window")

print("✓ Statistiques par capteur calculées")
print(f"\n📊 Schéma des statistiques par capteur :")
sensor_stats.printSchema()

## 6. Métriques métier (KPIs)

In [ ]:
# Calcul de métriques métier pour les tableaux de bord
# Agrégations globales par bâtiment (sans fenêtre temporelle pour l'instant)

# Pour les KPIs, on peut utiliser des agrégations sur fenêtres plus longues (quotidiennes)
daily_kpis = silver_with_watermark \
    .groupBy(
        window(col("timestamp"), "1 day"),  # Fenêtre quotidienne
        col("building_id")
    ) \
    .agg(
        # Consommation énergétique totale (KPI principal)
        sum("energy_consumption").alias("daily_total_energy"),
        avg("energy_consumption").alias("daily_avg_energy"),
        # Température moyenne
        avg("temperature").alias("daily_avg_temperature"),
        # Nombre de capteurs actifs
        countDistinct("sensor_id").alias("active_sensors"),
        # Nombre total de mesures
        count("*").alias("total_measurements"),
        # Taux d'anomalies
        (sum(when(col("anomaly_detected") == True, 1).otherwise(0)) / count("*") * 100).alias("anomaly_rate_percent")
    ) \
    .withColumn("date", to_date(col("window.start"))) \
    .withColumn("window_start", col("window.start")) \
    .withColumn("window_end", col("window.end")) \
    .drop("window")

print("✓ Métriques métier (KPIs) calculées")
print(f"\n📊 Schéma des KPIs :")
daily_kpis.printSchema()

## 7. Écriture dans Delta Lake Gold

In [ ]:
# Écriture des agrégations horaires dans Gold
# Mode Complete : réécrit la table complète à chaque micro-batch
# (nécessaire pour les agrégations avec fenêtres)

query_gold_hourly = hourly_stats \
    .writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{CHECKPOINT_GOLD_PATH}/hourly_stats") \
    .option("path", f"{DELTA_GOLD_PATH}/hourly_stats") \
    .partitionBy("building_id") \
    .trigger(processingTime="1 minute") \
    .start()

print("✓ Pipeline Gold (agrégations horaires) démarrée")
print(f"✓ Écriture dans : {DELTA_GOLD_PATH}/hourly_stats")
print(f"✓ Checkpoint dans : {CHECKPOINT_GOLD_PATH}/hourly_stats")
print(f"✓ Mode : Complete (réécriture complète pour agrégations)")
print(f"✓ Trigger : toutes les 1 minute")

In [ ]:
# Écriture des statistiques par capteur
query_gold_sensors = sensor_stats \
    .writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{CHECKPOINT_GOLD_PATH}/sensor_stats") \
    .option("path", f"{DELTA_GOLD_PATH}/sensor_stats") \
    .partitionBy("building_id", "sensor_type") \
    .trigger(processingTime="1 minute") \
    .start()

print("✓ Pipeline Gold (statistiques capteurs) démarrée")
print(f"✓ Écriture dans : {DELTA_GOLD_PATH}/sensor_stats")

In [ ]:
# Écriture des KPIs quotidiens
query_gold_kpis = daily_kpis \
    .writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{CHECKPOINT_GOLD_PATH}/daily_kpis") \
    .option("path", f"{DELTA_GOLD_PATH}/daily_kpis") \
    .partitionBy("building_id") \
    .trigger(processingTime="5 minutes") \
    .start()

print("✓ Pipeline Gold (KPIs quotidiens) démarrée")
print(f"✓ Écriture dans : {DELTA_GOLD_PATH}/daily_kpis")

## 8. Monitoring et vérification

In [ ]:
# Attendre quelques micro-batches pour voir les données
import time

print("Pipeline Gold en cours d'exécution...")
print("Attente de 60 secondes pour traiter les données...")

time.sleep(60)

# Vérifier le statut
print(f"\n✓ Statut query horaires : {query_gold_hourly.status}")
print(f"✓ Statut query capteurs : {query_gold_sensors.status}")
print(f"✓ Statut query KPIs : {query_gold_kpis.status}")

In [ ]:
# Vérifier les données écrites dans Delta Lake Gold
print("📊 Vérification des données dans Delta Lake Gold...")

try:
    # Agrégations horaires
    hourly_gold = spark.read.format("delta").load(f"{DELTA_GOLD_PATH}/hourly_stats")
    hourly_count = hourly_gold.count()
    
    if hourly_count > 0:
        print(f"\n✓ Agrégations horaires : {hourly_count} enregistrements")
        print("\n📋 Aperçu des agrégations horaires :")
        hourly_gold.orderBy(col("window_start").desc()).show(10, truncate=False)
    
    # Statistiques par capteur
    sensor_gold = spark.read.format("delta").load(f"{DELTA_GOLD_PATH}/sensor_stats")
    sensor_count = sensor_gold.count()
    
    if sensor_count > 0:
        print(f"\n✓ Statistiques capteurs : {sensor_count} enregistrements")
        print("\n📋 Aperçu des statistiques par capteur :")
        sensor_gold.orderBy(col("window_start").desc()).show(10, truncate=False)
    
    # KPIs quotidiens
    kpis_gold = spark.read.format("delta").load(f"{DELTA_GOLD_PATH}/daily_kpis")
    kpis_count = kpis_gold.count()
    
    if kpis_count > 0:
        print(f"\n✓ KPIs quotidiens : {kpis_count} enregistrements")
        print("\n📋 Aperçu des KPIs :")
        kpis_gold.orderBy(col("date").desc()).show(10, truncate=False)
    
    if hourly_count == 0 and sensor_count == 0 and kpis_count == 0:
        print("⚠️  Aucune donnée trouvée dans Gold")
        print("💡 Vérifiez que Silver contient des données et que les pipelines fonctionnent")
        
except Exception as e:
    print(f"❌ Erreur lors de la lecture : {e}")

## 9. Concepts : Windowing et Watermarks

### Windowing (Fenêtres temporelles)

Les **fenêtres temporelles** permettent de regrouper les données par période de temps pour les agrégations.

**Types de fenêtres** :
- **Fenêtre fixe** : `window(timestamp, "1 hour")` - fenêtres de 1 heure
- **Fenêtre glissante** : `window(timestamp, "1 hour", "30 minutes")` - fenêtre de 1h, slide de 30min

**Dans ce notebook** :
- Agrégations horaires : fenêtres fixes de 1 heure
- Statistiques capteurs : fenêtres glissantes (1h, slide 30min)
- KPIs quotidiens : fenêtres fixes de 1 jour

### Watermarks

Les **watermarks** permettent de gérer les données tardives (late data).

**Fonctionnement** :
- Définit un seuil de retard acceptable (ex: "1 hour")
- Les données arrivant après le watermark sont ignorées
- Permet de libérer la mémoire pour les anciennes fenêtres

**Dans ce notebook** :
- Watermark de 1 heure : accepte les données jusqu'à 1h de retard
- Permet d'utiliser Append mode avec des agrégations sur fenêtres

### Mode Complete pour agrégations

Pour les agrégations avec fenêtres temporelles, on utilise **Complete mode** car :
- Les fenêtres peuvent se chevaucher
- Les résultats doivent être réécrits complètement à chaque micro-batch
- Permet de maintenir l'état complet des agrégations

## 10. Arrêt propre des pipelines

In [ ]:
# Arrêter toutes les queries Gold
query_gold_hourly.stop()
query_gold_sensors.stop()
query_gold_kpis.stop()

print("✓ Pipelines Gold arrêtées proprement")

# Afficher un résumé final
try:
    hourly_final = spark.read.format("delta").load(f"{DELTA_GOLD_PATH}/hourly_stats")
    sensor_final = spark.read.format("delta").load(f"{DELTA_GOLD_PATH}/sensor_stats")
    kpis_final = spark.read.format("delta").load(f"{DELTA_GOLD_PATH}/daily_kpis")
    
    print(f"\n✓ Total agrégations horaires : {hourly_final.count()}")
    print(f"✓ Total statistiques capteurs : {sensor_final.count()}")
    print(f"✓ Total KPIs quotidiens : {kpis_final.count()}")
    print("\n✓ Pipeline Gold terminée avec succès !")
except Exception as e:
    print(f"⚠️  Erreur lors de la vérification finale : {e}")